# Agentic AI Research Agent
**Flow:** User Query → Query Rewrite → Tavily Web Search → LLM Synthesizer → Critic → (retry?) → **Human Approval** → (approved → Done) | (rejected → Query Rewrite with feedback)

In [ ]:
!pip install -q langchain langchain-openai langgraph langchain-tavily python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 20.0 MB/s eta 0:00:00


In [ ]:
# ── CELL 1: Imports & API Keys ──────────────────────────────────────────────
# Store your keys in a .env file (never hardcode them!):
#   TAVILY_API_KEY=tvly-...
#   GROQ_API_KEY=gsk_...

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END

load_dotenv()

import os

os.environ["GROQ_API_KEY"] = "gsk_ah"
os.environ["TAVILY_API_KEY"] = "tvly-d"

print("✅ API keys loaded successfully.")

✅ API keys loaded successfully.


In [20]:
# ── CELL 2: State ────────────────────────────────────────────────────────────
class ResearchState(TypedDict):
    query:          str # This stores the original question from the user.
    web_results:    str # Stores everything returned from Tavily search.
    search_query:   str # Stores the rewritten query.
    final_answer:   str # Stores the answer generated by the LLM.
    needs_retry:    bool # A Boolean (True or False) indicating whether the graph should try again.
                         # For example, the critic node may determine: This answer is incomplete. (then this is true)
    retry_count:    int  # Tracks how many retry attempts have been made.
    human_approved: bool # Stores whether the human approved the answer. (intially false)
    human_feedback: str # Stores any feedback from the human.

In [21]:
# ── CELL 3: Tools & LLM ──────────────────────────────────────────────────────
tavily = TavilySearch(max_results=5)

llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [22]:
# Query Rewrite
# If human_feedback exists (i.e. human rejected previous answer),
# it's included so the rewrite is smarter — not just the same query again.

REWRITE_PROMPT = """\
You rewrite user questions into a short, high-signal web search query.
Return ONLY the search query text. No explanation, no quotes.
"""

def query_rewrite_node(state: ResearchState) -> dict: # this is how u should write the function

    feedback = state.get("human_feedback", "").strip() # taking the human feedback from state and storing it in a variable

    if feedback: # in this if we have a user feedback the we are making it better for llm so he can understand in a good way
        user_content = (
            f"Original question: {state['query']}\n" #user content mai main query ko bhi daal rahi hai
            f"Previous answer was rejected. Human feedback: {feedback}\n" # human feedback ko bhi send kar rahe hai
            f"Rewrite the search query to address this feedback." # and special command to rewrite the query
        )
        print(f"    Using human feedback to improve query: '{feedback}'")
    else:
        user_content = state["query"] # else we are just sending the query llm will improve according to it .

    messages = [ # learn this is the way to store the user message , system message
        SystemMessage(content=REWRITE_PROMPT),
        HumanMessage(content=user_content)
    ]
    response = llm.invoke(messages) # calling the llm
    rewritten = response.content.strip()

    print(f"    Search query: {rewritten}")
    return {"search_query": rewritten}

In [23]:
# ── CELL 5: Node 2 — Web Search ──────────────────────────────────────────────
def web_search_node(state: ResearchState) -> dict:

    raw = tavily.invoke({"query": state["search_query"]}) # here we have search query not query bec in the firsl flow we are directly
                                                          # sending the query to web search and when human rejects then we are using this .
    raw = raw["results"] # Now instead of the whole response, you keep only the list.(@)

    formatted_results = [] # This will store nicely formatted text.

    for i, result in enumerate(raw, start=1): # Loop through every result
        title   = result.get("title",   "No title") # is line ke niche se(we are filling the details in formated results )
        url     = result.get("url",     "")
        content = result.get("content", "").strip()
        formatted_results.append(
            f"[{i}] {title}\n"
            f"    URL: {url}\n"
            f"    {content}"
        )

    web_results_text = "\n\n".join(formatted_results) # join() combines all list items into one string.

    return {"web_results": web_results_text}

In [24]:
# LLM Synthesizer  => This is the LLM Synthesizer node. Its job is not to search the web. The search has already been done in the previous node.
                      #This node takes those search results and asks the LLM to write a final, readable answer.

SYSTEM_PROMPT = """\
You are an expert research assistant.

You will be given:
  1. A user research query.
  2. Five web search results (title, URL, snippet).

Your job:
  - Synthesize the web results with your own knowledge.
  - Write a clear, well-structured, comprehensive answer.
  - Cite sources inline using [1], [2], etc. where relevant.
  - Highlight any conflicting information across sources.
  - End with a "Sources" section listing all URLs.
  - Be factual. Do not hallucinate beyond the provided data.
"""

def synthesizer_node(state: ResearchState) -> dict:

    user_prompt = (
        f"Research Query: {state['query']}\n\n"
        f"Web Search Results:\n\n{state['web_results']}"
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT), # system message mai prompt send kar rahe what he have to do
        HumanMessage(content=user_prompt) # from state we are sending the reseach query
    ]

    response = llm.invoke(messages)

    return {"final_answer": response.content}

In [25]:
#  Critic
CRITIC_PROMPT = """\
You judge whether a research answer is well supported by the given sources.
Reply with exactly one word: "yes" if well supported,
"no" if it is thin, vague, or barely uses the sources.
"""

def critic_node(state: ResearchState) -> dict:

    user_prompt = (
        f"Sources:\n{state['web_results']}\n\n"
        f"Answer:\n{state['final_answer']}"
    )

    messages = [
        SystemMessage(content=CRITIC_PROMPT),
        HumanMessage(content=user_prompt)
    ]

    verdict = llm.invoke(messages).content.strip().lower()

    if verdict.startswith("no") and state["retry_count"] < 3:
        needs_retry = True
    else:
        needs_retry = False

    return {
        "needs_retry": needs_retry,
        "retry_count": state["retry_count"] + 1
    }

In [26]:
#  Human Approval
# Uses interrupt() to pause graph and wait for human input.
# If human types 'ok' → approved → END
# If human types anything else → treated as feedback → loops back to query_rewrite

def human_approval_node(state: ResearchState) -> dict:
    from langgraph.types import interrupt


    print(" HUMAN APPROVAL REQUIRED : \n")

    preview = state["final_answer"][:600] # this line take the starting 600 charcater and terminate the rest

    if len(state["final_answer"]) > 600: # this if statement will add "..." at the end of the answer to there was something after this
        preview += "..."

    print(preview)
    print()

    # Pause here — resume value becomes user_input
    user_input = interrupt("Type 'ok' to approve, or give feedback to improve the answer: ") # interrupt function to pause the follow of the graph and to take the input also (@)

    approved = user_input.strip().lower() == "ok"

    if approved:
        print("✅ Approved!")
        return {
            "human_approved": True,
            "human_feedback": ""
        }
    else:
        feedback = user_input.strip()
        print(f"🔄 Rejected. Feedback: '{feedback}'. Retrying with improved query...")
        return {
            "human_approved": False,
            "human_feedback": feedback,
            "retry_count":    0
        }

In [27]:
# ── CELL 9: Build Graph ───────────────────────────────────────────────────────
#
#   START → query_rewrite → web_search → synthesizer → critic
#               ↑                                         |
#               └──── needs_retry=True ───────────────────┘
#                              (else)
#                                ↓
#                        human_approval   ← graph PAUSES here
#                        /           \
#                  approved         rejected (with feedback)
#                     ↓                  ↓
#                    END          query_rewrite  (smarter retry)

from langgraph.checkpoint.memory import MemorySaver

def route_after_critic(state: ResearchState):
    return "web_search" if state["needs_retry"] else "human_approval"

# FIX 1: Use string key "end" instead of END constant as a dict key —
# using END directly as a dict key causes a runtime crash.
def route_after_human(state: ResearchState):
    return "end" if state["human_approved"] else "query_rewrite"


workflow = StateGraph(ResearchState)

workflow.add_node("query_rewrite",  query_rewrite_node)
workflow.add_node("web_search",     web_search_node)
workflow.add_node("synthesizer",    synthesizer_node)
workflow.add_node("critic",         critic_node)
workflow.add_node("human_approval", human_approval_node)

workflow.add_edge(START,           "query_rewrite")
workflow.add_edge("query_rewrite", "web_search")
workflow.add_edge("web_search",    "synthesizer")
workflow.add_edge("synthesizer",   "critic")

workflow.add_conditional_edges(
    "critic",
    route_after_critic,
    {"web_search": "web_search", "human_approval": "human_approval"}
)

# FIX 1 continued: "end" string maps to END constant here — this is correct
workflow.add_conditional_edges(
    "human_approval",
    route_after_human,
    {"end": END, "query_rewrite": "query_rewrite"}
)

memory = MemorySaver()
agent  = workflow.compile(checkpointer=memory)  # checkpointer laga hai na ?


✅ Graph compiled successfully.


In [28]:
# ── CELL 10: Run Function ─────────────────────────────────────────────────────
# Handles the multi-phase execution:
#   Phase 1 — runs until graph pauses at human_approval
#   Phase 2+ — resumes with human input; loops if rejected until approved

def research(query: str):
    import uuid

    thread = {
        "configurable": {
            "thread_id": str(uuid.uuid4()) # Generates a random unique identifier.
        }
    }

    print(f"\nQuery: {query}\n")

    # Phase 1: Run until first interrupt
    agent.invoke(
        {
            "query": query,
            "search_query": "",
            "web_results": "",
            "final_answer": "",
            "needs_retry": False,
            "retry_count": 0,
            "human_approved": False,
            "human_feedback": "",
        },
        config=thread, # evry excution have its own thread
    )

    # Read state after interrupt
    state = agent.get_state(thread).values # this line retrive the saved state

    # Phase 2+: Human review loop
    while not state.get("human_approved", False): # Continue looping while human_approved is False.

        print(state["final_answer"]) # The generated answer is displayed to the user.
        print()

        human_input = input("Type 'ok' to approve, or give feedback: ") # Get the user's response

        # Resume graph with human input
        agent.invoke(
            {"resume": human_input},
            config=thread,
        )

        # Read updated state
        state = agent.get_state(thread).values

    print("\nFinal Approved Answer:\n")
    print(state["final_answer"])

    return state

In [33]:
# ── CELL 11: Test It ──────────────────────────────────────────────────────────
research("What are the latest developments in quantum computing in 2025?")  # rate limit error = It means you've made too many API requests in a short period of time.


 QUERY: What are the latest developments in quantum computing in 2025?

    Search query: quantum computing advancements 2025


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktny3ga3fg8a35vk8t68r24w` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 97901, Requested 4611. Please try again in 36m10.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [32]:
# ── CELL 12: Try Your Own Query ───────────────────────────────────────────────
research("Impact of AI on software engineering jobs")


 QUERY: Impact of AI on software engineering jobs

    Search query: AI impact on software engineering careers


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktny3ga3fg8a35vk8t68r24w` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99591, Requested 2182. Please try again in 25m31.871999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}